# CPSC 375 Classwork - K-Nearest Neighbors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProfAvery/cpsc375-classwork/blob/main/knn.ipynb)

- *Manager*: Sepehr Kermanshah
- *Recorder*: Bradley Botros
- *Presenter*:
- *Timekeeper*:

*Note*: On Google Colab, run the following code cell to download the `cpsc375` module before trying to `import`:

In [1]:
!mkdir -p ./cpsc375 && wget --quiet --timestamping https://raw.githubusercontent.com/ProfAvery/cpsc375-classwork/refs/heads/main/cpsc375/__init__.py --output-document=./cpsc375/__init__.py

In [2]:
from cpsc375 import *

## Part 1 - Slide 22

* Compute the Euclidean distance between vectors
  - (5, 0, -1)
  - (7, 3,  5)
* Do it by hand
* Check with Python

d = squre_root((7−5)^2 + (3−0)^2 + (5−(−1))^2)
  = square_root(2^2 + 3^2 + 6^2)
  = square_root(4 + 9 + 36)
  = square_root49
  = 7

In [4]:
distance.euclidean((5, 0, -1), (7, 3, 5))

np.float64(7.0)

In [3]:
?distance.euclidean


* Consider this training data

|          | nchars | is_last_a | nvowels | **Output** |
|----------|--------|-----------|---------|------------|
| "Laura"  | 5      | 1         | 3       | **F**      |
| "Carlos" | 6      | 0         | 2       | **M**      |
| "Jose"   | 4      | 0         | 2       | **M**      |
| "Maria"  | 5      | 1         | 3       | **F**      |
| "Belen"  | 5      | 0         | 2       | **F**      |

`is_last_a`: 1 if last character is "a", 0 otherwise

* Using 1-NN, what is the predicted output name for "Chris"?


"Chris" -> nchars = 5, is_last_a = 0, nvowels = 1 -> (5, 0, 1)

| Name | Vector | Distance to Chris | Output |
|---|---|---|---|
| Laura | (5, 1, 3) | square_root5 about 2.24 | F |
| Carlos | (6, 0, 2) | square_root2 about 1.41 | M |
| Jose | (4, 0, 2) | square_root 2 about 1.41 | M |
| Maria | (5, 1, 3) | square_root5 about 2.24 | F |
| Belen | (5, 0, 2) | 1 | F |

Nearest neighbor is Belen (distance 1), so the 1 NN prediction is F.



* Using 3-NN, what is the predicted output name for "Chris"?


The 3 nearest neighbors are Belen (F), Carlos (M), and Jose (M).

Majority vote: 2 M vs 1 F, so the 3 NN prediction is M.

## Part 2 - Slide 27

The `pima-indians-diabetes-resampled.csv` file on Canvas contains a version of the Pima Indian Diabetes Database (PIDD). This dataset is often used to test machine learning algorithms, but it was not collected in a vacuum, and it is important to acknowledge [the background and the ethics issues it raises](https://nihrecord.nih.gov/2020/07/24/medical-history-matters-era-big-data). The subjects of the database are of Pima Indian heritage, and the dataset was originally collected as part of a longitudinal study of diabetes by the National Institutes of Health. The records in the dataset indicate whether the subjects have diabetes or not, along with certain diagnostic measurements.


In [5]:
pima_indians_diabetes = pd.read_csv(
    "https://raw.githubusercontent.com/ProfAvery/cpsc375-classwork/refs/heads/main/datasets/pima-indians-diabetes-resampled.csv"
)

The goal is to predict if a subject has diabetes, given some of the diagnostic measurements (only *Preg* and *Pedigree*)

* Which is the “class” variable? Which is/are the “feature” variables?

**Class variable:** HasDiabetes

**Feature variables:** Preg and Pedigree

* Split the dataset into train and test datasets with the *first 500 rows* for training, and the remaining rows for test
  - Do NOT randomly sample the data


In [7]:
train = pima_indians_diabetes.iloc[:500]
test = pima_indians_diabetes.iloc[500:]

features = ["Preg", "Pedigree"]
X_train, y_train = train[features], train["HasDiabetes"]
X_test, y_test = test[features], test["HasDiabetes"]

* Train and test a 1-nearest neighbor (1-NN) classifier with this dataset
  - What is the error rate (number of misclassifications)?


In [8]:
mod = KNeighborsClassifier(n_neighbors=1)
mod.fit(X_train, y_train)

predictions = mod.predict(X_test)
misclassified = (predictions != y_test).sum()
print("Misclassifications:", misclassified, "out of", len(y_test))
print("Error rate:", 1 - mod.score(X_test, y_test))

Misclassifications: 108 out of 268
Error rate: 0.4029850746268657


The 1 NN classifier misclassified 108 out of 268 test rows, so the error rate is about **40.3%**.

## Part 3 - Slide 45

|   |                                     |            |
|---|-------------------------------------|------------|
| A | { Apple, Banana, Soymilk, Yoghurt } | Vegetarian |
| B | { Apple, Peanuts, Yoghurt }         | Vegetarian |
| C | { Tomatoes, Potatoes, Yoghurt }     | Meat-eater |
| D | { Apple, Tomatoes, Potatoes }       | Meat-eater |

Use 1-NN algorithm to predict

|   |                                     |            |
|---|-------------------------------------|------------|
| E | { Apple, Tomatoes }                  | ?          |
| F | { Yoghurt, Soymilk }                 | ?          |


| | Similarity |
|---|---|
| J(E, A) | 1/5 = 0.20 |
| J(E, B) | 1/4 = 0.25 |
| J(E, C) | 1/4 = 0.25 |
| J(E, D) | 2/3 = 0.67 |

Predicted class for E: Meat eater (D is the nearest neighbor)

| | Similarity |
|---|---|
| J(F, A) | 2/4 = 0.50 |
| J(F, B) | 1/4 = 0.25 |
| J(F, C) | 1/4 = 0.25 |
| J(F, D) | 0/5 = 0 |

Predicted class for F: Vegetarian (A is the nearest neighbor)
